# 53 Veeva Link config tables v1 (Phase 0, U4)

Builds the three config tables that fix the Veeva block schema (plan `51_`, section 4):

* **(a) Focus-name dictionary**: read verbatim from assessment v2 section 5.3 (`40_`): 17 `CLINICAL_LEADER`, 147 `FOCUS_AREA`, 135 `DIGITAL_REACH_FOCUS_AREA` names with theme and indication. Checked against the frozen snapshot (`52_`).
* **(b) Free-text keyword list**: the `LIKE` patterns of `33_` section 7, one row per pattern and tag, plus a talk-title stop-list.
* **(c) Window list**: 3 and 5 years.

Coverage (A5, 147,489 HCPs) needs the spine and is checked in `54_`.

In [0]:
from pyspark.sql import functions as F
import pandas as pd, re

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAPSHOT_ID = "load=2025-06-30"
SNAP = f"{OUT}snapshot/{SNAPSHOT_ID}/"
CFG = f"{OUT}config/"
DOCS = "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_1_develop_sm/NBA 3.1 - Dev/Veeva Link To People Analysis/"
ASSESS, PLAN33 = DOCS + "40_veeva_link_data_assessment_v2.md", DOCS + "33_veeva_link_feature_engineering_plan.md"

# Theme -> indication view (plan 50, principle 6: Kerendia total plus HF / T1D / CKD views)
INDICATION = {"CKD": "CKD", "Heart failure": "HF", "T1D": "T1D"}

In [0]:
lines = open(ASSESS, encoding="utf-8").read().splitlines()
i_cl = next(i for i, l in enumerate(lines) if l.startswith("#### Focus names used") and "CLINICAL_LEADER" in l)
i_fa = next(i for i, l in enumerate(lines) if l.startswith("#### Focus names used") and "FOCUS_AREA" in l)
i_end = next(i for i, l in enumerate(lines) if l.startswith("#### Coverage within the overlap"))

rows = []
for l in lines[i_cl:i_fa]:                       # | TYPE | Theme | FOCUS_NAME | n |
    if l.startswith("| disease") or l.startswith("| drug_class"):
        p = [x.strip() for x in l.split("|")]
        rows.append({"source_table": "clinical_leader", "cl_type": p[1], "theme": p[2], "focus_name": p[3]})
for l in lines[i_fa:i_end]:                      # | Theme (n) | name; name*; name† |
    if l.startswith("|") and "(" in l.split("|")[1] and not l.startswith("| Theme") and not l.startswith("|---"):
        p = l.split("|")
        theme = re.sub(r"\s*\(\d+\)\s*$", "", p[1].strip())
        for n in p[2].split("; "):
            n = n.strip()
            only_fa, only_dr = n.endswith("*"), n.endswith("\u2020")
            n = n.rstrip("*\u2020").strip()
            if not only_dr: rows.append({"source_table": "focus_area", "cl_type": None, "theme": theme, "focus_name": n})
            if not only_fa: rows.append({"source_table": "digital_reach_focus_area", "cl_type": None, "theme": theme, "focus_name": n})

fd = pd.DataFrame(rows).drop_duplicates(["source_table", "cl_type", "focus_name"])
fd["indication"] = fd.theme.map(INDICATION).fillna("other")
fd["kerendia_relevant"] = 1
fd["dictionary_version"] = "v1"
fd["source_doc"] = "40_veeva_link_data_assessment_v2.md section 5.3"
print(fd.groupby("source_table").focus_name.nunique().to_dict(), "(expected clinical_leader 17, focus_area 147, digital_reach_focus_area 135)")
display(fd.groupby(["source_table", "theme", "indication"]).size().reset_index(name="n_names"))

In [0]:
chk = []
for t in ["clinical_leader", "focus_area", "digital_reach_focus_area"]:
    keys = ["FOCUS_NAME", "TYPE"] if t == "clinical_leader" else ["FOCUS_NAME"]
    snap_names = spark.read.parquet(f"{SNAP}{t}").select(*keys).distinct().toPandas()
    d = fd[fd.source_table == t]
    if t == "clinical_leader":
        m = d.merge(snap_names, left_on=["focus_name", "cl_type"], right_on=["FOCUS_NAME", "TYPE"], how="left", indicator=True)
    else:
        m = d.merge(snap_names, left_on="focus_name", right_on="FOCUS_NAME", how="left", indicator=True)
    missing = m[m._merge == "left_only"].focus_name.tolist()
    chk.append({"table": t, "names": len(d), "found_in_snapshot": int((m._merge == "both").sum()), "missing": missing})
    if t == "focus_area":   # carry FOCUS_TYPE onto the dictionary
        ft = spark.read.parquet(f"{SNAP}{t}").filter(F.col("FOCUS_NAME").isin(d.focus_name.tolist())).select("FOCUS_NAME", "FOCUS_TYPE").distinct().toPandas()
chk = pd.DataFrame(chk); display(chk)
fd = fd.merge(ft.groupby("FOCUS_NAME").FOCUS_TYPE.agg(lambda s: "; ".join(sorted(s))).rename("focus_type"),
              left_on="focus_name", right_index=True, how="left")
fd.loc[fd.source_table != "focus_area", "focus_type"] = None
print("FOCUS_AREA names by FOCUS_TYPE:", fd[fd.source_table == "focus_area"].focus_type.value_counts().to_dict(), "(assessment: Drugs 92, Diseases 39, Diagnosis and Treatment 9, Biology 7)")

In [0]:
L33 = open(PLAN33, encoding="utf-8").read().splitlines()
s = next(i for i, l in enumerate(L33) if l.startswith("## 7"))
e = next(i for i, l in enumerate(L33) if i > s and l.startswith("## "))
sec = L33[s:e]
k0 = next(i for i, l in enumerate(sec) if l.startswith("### ILIKE patterns"))   # skip the REGEXP block (FOCUS_NAME only)

TAGS = {"T2D": "t2d", "CKD": "ckd", "T1D": "t1d", "HF": "hf", "Kerendia / finerenone": "finerenone", "MRAs": "mra",
        "SGLT2 inhibitors": "sglt2", "GLP-1 receptor agonists": "glp1", "RAAS pathway": "raas",
        "Endothelin receptor antagonists": "era", "CKD/cardiorenal therapeutic area terms": "cardiorenal",
        "Landmark cardiorenal trial names": "landmark_trial"}
kw, tag = [], None
for l in sec[k0:]:
    if l.strip().startswith("--") and not l.strip().startswith("---"):   # SQL comment header, not a markdown rule
        label = l.strip()[2:].strip()
        tag = next((TAGS[k] for k in sorted(TAGS, key=len, reverse=True) if label.startswith(k)), None)  # longest key first ('CKD/cardiorenal...' before 'CKD')
        assert tag, f"unmapped block: {label}"
    for pat in re.findall(r"LIKE '([^']+)'", l):
        kw.append({"tag": tag, "like_pattern": pat})
kwd = pd.DataFrame(kw).drop_duplicates()
GROUP = {"t2d": "indication", "ckd": "indication", "t1d": "indication", "hf": "indication"}
kwd["tag_group"] = kwd.tag.map(GROUP).fillna("drug_or_trial")
kwd["applies_to"] = "PUBLICATION.TITLE; CLINICAL_TRIAL.NAME; MEDICAL_EVENT.NAME; EVENT_ATTENDEE.TALK_TITLE"
kwd["match_rule"] = "lower(col) LIKE like_pattern (Spark, on parquet)"
kwd["dictionary_version"] = "v1"
print("patterns:", len(kwd)); display(kwd.groupby(["tag_group", "tag"]).size().reset_index(name="n_patterns"))

# Generic session labels that carry no topic (assessment 4.5); matched after lower/trim
stop = pd.DataFrame({"talk_title_stop": ["introduction", "panel discussion", "welcome", "welcome and introduction",
                     "opening remarks", "closing remarks", "discussion", "q&a", "questions and answers", "break", "lunch"],
                     "rule": "exact"})
stop = pd.concat([stop, pd.DataFrame({"talk_title_stop": [r"^session \d+$"], "rule": "regex"})], ignore_index=True)
ev = spark.read.parquet(f"{SNAP}event_attendee").select(F.lower(F.trim("TALK_TITLE")).alias("t"))
exact = stop[stop.rule == "exact"].talk_title_stop.tolist()
print("talk titles hit by stop-list:", ev.filter(F.col("t").isin(exact) | F.col("t").rlike(r"^session \d+$")).count(), "of", ev.count())

In [0]:
win = pd.DataFrame([
    {"window": "3y", "months": 36, "families": "events; payments", "rule": "window ends at min(anchor month, table data cut-off)"},
    {"window": "5y", "months": 60, "families": "trials; publications", "rule": "window ends at min(anchor month, table data cut-off)"},
]).assign(dictionary_version="v1")

for name, df in [("focus_dictionary_v1", fd), ("keyword_list_v1", kwd), ("talk_title_stoplist_v1", stop), ("windows_v1", win)]:
    spark.createDataFrame(df.astype(object).where(df.notna(), None)).write.mode("overwrite").parquet(CFG + name)
    print(name, spark.read.parquet(CFG + name).count(), "rows")

In [0]:
# v2 = every v1 include pattern, plus exclude patterns for the MRA tag (spot-check audit, 70_ section 8, 2026-11-23).
# Rule: a text keeps TAG_MRA only if it matches an MRA include pattern AND no MRA exclude pattern.
# Targets non-cardiorenal spironolactone uses: androgen excess / acne / hirsutism / PCOS, alopecia, transgender care, PAH.
MRA_EXCLUDE = ["%acne%", "%androgen%", "%hirsut%", "%polycystic ovar%", "%pcos%", "%alopecia%", "%hair loss%",
               "%transgender%", "%gender-affirming%", "%gender affirming%", "%feminiz%", "%dermatolog%",
               "%pulmonary arterial hypertension%"]
v1 = spark.read.parquet(CFG + "keyword_list_v1").toPandas()
inc = v1.assign(rule_type="include")
exc = pd.DataFrame({"tag": "mra", "like_pattern": MRA_EXCLUDE}).assign(
    tag_group="drug_or_trial", applies_to=v1.applies_to.iloc[0],
    match_rule="tag kept only if no exclude pattern for the same tag matches: lower(col) NOT LIKE like_pattern", rule_type="exclude")
kwd2 = pd.concat([inc, exc], ignore_index=True).assign(dictionary_version="v2")
kwd2 = kwd2[["tag", "like_pattern", "rule_type", "tag_group", "applies_to", "match_rule", "dictionary_version"]]
assert len(inc) == len(v1) and kwd2.duplicated(["tag", "like_pattern", "rule_type"]).sum() == 0
spark.createDataFrame(kwd2).write.mode("overwrite").parquet(CFG + "keyword_list_v2")
print("keyword_list_v2:", spark.read.parquet(CFG + "keyword_list_v2").count(), "rows (", len(inc), "include +", len(exc), "exclude )")

# Impact on the current activity tables (tags built with v1 in 55_): rows whose TAG_MRA would flip to 0
ACT = f"{OUT}activity/"
from functools import reduce
excl = reduce(lambda a, b: a | b, [F.col("_txt").like(p) for p in MRA_EXCLUDE])
impact = []
for fam, txt in [("trials", F.lower("NAME")), ("publications", F.lower("TITLE")), ("events", F.lower(F.concat_ws(" ", "EVENT_NAME", "TALK_TITLE")))]:
    m = spark.read.parquet(ACT + fam).filter("TAG_MRA = 1").withColumn("_txt", txt).withColumn("flip", excl.cast("int"))
    a = m.agg(F.count("*").alias("mra_rows"), F.sum("flip").alias("rows_flipped"),
              F.countDistinct("SE_ID").alias("experts_with_mra")).first().asDict()
    a["experts_losing_mra"] = m.groupBy("SE_ID").agg(F.min("flip").alias("all_off")).filter("all_off = 1").count()
    impact.append({"family": fam, **a})
display(pd.DataFrame(impact))
print("Not yet applied: 55_ still tags with keyword_list_v1. Switching 55_ to v2 and re-running 55_ -> 56_ -> 71_ is a separate step.")